<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Jour 1b — Modèles de référence : k-mer + ML classique, et one-hot + CNN
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Représenter une séquence d'ADN par des fréquences de k-mers (k=4) et entraîner une régression logistique<br>
    - Encoder les nucléotides en one-hot et entraîner un petit CNN 1D qui apprend ses propres motifs<br>
    - Comparer les deux références sur accuracy, F1, nombre de paramètres et latence<br>
</div>

#### **Votre identité**

Double-cliquez sur cette cellule et complétez, puis exécutez-la (`Maj + Entrée`).

- **Nom & prénom :** BOSSA Chabel
- **Groupe / binôme :** _à compléter_
- **Date :** _à compléter_

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

Deux façons de transformer une chaîne d'ADN en quelque chose qu'un modèle peut utiliser,
avant même de toucher à un modèle de langage pré-entraîné :

1. **Fréquences de k-mers** : compter la fréquence de chaque sous-chaîne de longueur k
   (par ex. les 256 4-mers possibles), normaliser -> un vecteur de taille fixe -> ML
   classique (régression logistique).
2. **One-hot + CNN** : encoder chaque nucléotide en un vecteur one-hot de dimension 4
   (A/C/G/T), empiler en un tenseur (4, 200), et laisser un petit CNN 1D apprendre
   lui-même ses propres caractéristiques directement à partir de la séquence brute.

<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day1/assets/illustration2.png"/>

In [1]:
import sys
sys.path.append("src")

import numpy as np
import torch
from data import load_all # importer depuis le dossier src/data.py

# N_WINDOWS fenêtres par split, équilibrées codant/non-codant : le MÊME budget
# que les embeddings Evo2 du Jour 2, pour que la comparaison du Jour 4 soit juste
N_WINDOWS = 4000
splits = load_all("../2-data/processed", max_rows=N_WINDOWS)
train, val = splits["train"], splits["val"]

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Modèle de référence 1 : fréquences de k-mers + régression logistique**

**À faire :** 
- complétez le code ci-dessous pour calculer les matrices de k-mers (k=4) sur
train/val, 
- entraîner un classifieur logistique (`make_kmer_classifier("logreg")`), 
- puis l'évaluer sur validation avec `evaluate_sklearn`.

In [2]:
from featurize import kmer_matrix  # importer depuis le dossier src/featurize.py

# Si train et val ne sont pas encore définis à partir du dictionnaire splits :
train = splits["train"]
val = splits["val"]

K = 4
# Calcul des matrices de k-mers (4-mers = 256 caractéristiques par séquence)
X_train_kmer = kmer_matrix(train["sequence"], k=K)
X_val_kmer = kmer_matrix(val["sequence"], k=K)

# Extraction des étiquettes (0 = non-codant, 1 = codant)
y_train, y_val = train["label"].to_numpy(), val["label"].to_numpy()

In [3]:
from models.baselines import make_kmer_classifier  # importer depuis le dossier src/models

# Créez un classifieur avec make_kmer_classifier("logreg") et entraînez-le (fit)
kmer_clf = make_kmer_classifier("logreg")
kmer_clf.fit(X_train_kmer, y_train)

# Classifieur Arbre de décision (Decision Tree)
kmer_dt_clf = make_kmer_classifier("dt")
kmer_dt_clf.fit(X_train_kmer, y_train)


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at

In [4]:
from eval import evaluate_sklearn # importer depuis le dossier src/eval.py

# Évaluez la régression logistique et l'arbre de décision sur les données de validation
kmer_metrics = evaluate_sklearn(kmer_clf, X_val_kmer, y_val)
kmer_dt_metrics = evaluate_sklearn(kmer_dt_clf, X_val_kmer, y_val)
print("k-mer + logreg:", kmer_metrics)
print("k-mer + decision tree:", kmer_dt_metrics)


k-mer + logreg: {'accuracy': 0.7885, 'f1': 0.807989105764866}
k-mer + decision tree: {'accuracy': 0.718, 'f1': 0.7287157287157288}


#### **Modèle de référence 2 : nucléotides one-hot + petit CNN**

**À faire :** encodez les fenêtres en one-hot (`one_hot_batch`, longueur 200), instanciez
`OneHotCNN`, puis entraînez-le pendant 5 époques (Adam, lr=1e-3,
`binary_cross_entropy_with_logits`, mini-lots de 256).

In [5]:
from featurize import one_hot_batch  # importer depuis le dossier src/featurize.py
from models.baselines import OneHotCNN        # importer depuis le dossier src/models
from eval import evaluate_logits, count_params, measure_latency_sklearn, measure_latency_torch # importer depuis le dossier src/eval.pyfe

In [ ]:
# N_EPOCHS : même budget d'entraînement pour TOUS les modèles de la semaine,
# pour que la comparaison du Jour 4 porte sur la représentation et non sur
# la durée d'entraînement. Modifiable ici.
N_EPOCHS = 100
WINDOW = 200
# TODO : encodez les fenêtres d'entraînement et de validation en one-hot, convertissez en tensors torch
X_train_oh = torch.tensor(one_hot_batch(train["sequence"], length=WINDOW), dtype=torch.float32)
X_val_oh = torch.tensor(one_hot_batch(val["sequence"], length=WINDOW), dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

# TODO : instanciez le modèle OneHotCNN(seq_len=WINDOW) et un optimiseur Adam (lr=1e-3)
cnn = OneHotCNN(seq_len=WINDOW) 
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()

n = X_train_oh.shape[0]

for epoch in range(N_EPOCHS):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        # TODO : calculez les logits du CNN sur ce mini-lot, puis la perte BCE-with-logits
        logits = cnn(X_train_oh[idx])
        loss = criterion(logits, y_train_t[idx])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"epoch {epoch+1}/{N_EPOCHS}: loss={epoch_loss/n:.4f}")

epoch 1/100: loss=0.6804
epoch 20/100: loss=0.4928
epoch 40/100: loss=0.3740
epoch 60/100: loss=0.3607
epoch 80/100: loss=0.3387
epoch 100/100: loss=0.3272


In [7]:
# TODO : passez X_val_oh dans le CNN (sans gradient) puis évaluez avec evaluate_logits
with torch.no_grad():
    val_logits = cnn(X_val_oh)
    print(f'val_logits {val_logits}')
cnn_metrics = evaluate_logits(val_logits, y_val_t)
print("one-hot CNN:", cnn_metrics)

val_logits tensor([-1.7242,  1.1615,  0.0650,  ..., -2.1760,  1.2839,  6.2788])
one-hot CNN: {'accuracy': 0.83325, 'f1': 0.8413038305971925}


#### **Comparaison**

In [8]:
import pandas as pd

# TODO : complétez le nombre de paramètres et la latence pour chaque modèle
# (count_params / measure_latency_sklearn pour le k-mer, count_params / measure_latency_torch pour le CNN)
comparison = pd.DataFrame([
    {"model": "kmer+logreg", **kmer_metrics,
     "params": 257,
     "latency_ms": measure_latency_sklearn(kmer_clf, X_val_kmer[:1])},
    {"model": "kmer+dt", **kmer_dt_metrics,
     "params": getattr(kmer_dt_clf, "tree_").node_count,
     "latency_ms": measure_latency_sklearn(kmer_dt_clf, X_val_kmer[:1])},
    {"model": "onehot+CNN", **cnn_metrics,
     "params": count_params(cnn),
     "latency_ms": measure_latency_torch(cnn, X_val_oh[:1] )},
])
comparison


,model,accuracy,f1,params,latency_ms
0,kmer+logreg,0.78850,0.807989,257,0.072532
1,kmer+dt,0.71800,0.728716,683,0.056826
2,onehot+CNN,0.83325,0.841304,10465,0.181185


#### **Point de contrôle**

Vous devriez avoir un tableau accuracy/F1/params/latence pour les deux modèles de
référence. Gardez ces chiffres (ou le DataFrame `comparison`) — ils serviront pour le
graphique d'efficacité du Jour 4.

Suite : `02_evo2_embeddings_and_classifier.ipynb` — un modèle de fondation génomique
fait-il mieux ?

*Bloqué ? La version complète est dans `solution/01_kmer_and_cnn_baselines.ipynb`.*

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Fin du Jour 1b</div>
      <div>Prochaine &eacute;tape &rarr; <code>day2/02_evo2_embeddings_and_classifier.ipynb</code></div>
    </div>
    <div style="flex: 0 0 auto; text-align: right; border-right: 2px solid #0969da; padding-right: 0.9em;">
      <div style="font-weight: 600; color: #24292f;">EEIA &middot; bioAI Workshop</div>
      <div style="font-size: 0.85em;">Semaine 4 &mdash; De l'ADN aux mod&egrave;les compress&eacute;s</div>
    </div>
  </div>
</div>